In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve,
    classification_report, confusion_matrix, average_precision_score
)
from sklearn.neighbors import BallTree
from catboost import CatBoostClassifier
import json
import joblib

In [2]:
DATA_PATH = "C:\GaTech\MS-CSE\ISYE 6740\WiFOP\data\\final datasets\ca_5km_daily_panel_2020_2023_6_9 1\ca_5km_daily_panel_2020_2023_6_9.csv"
MODEL_PATH = "../models/2020_2023/catboost_model.cbm"
THRESHOLD_PATH = "../models/2020_2023/catboost_threshold.json"
META_PATH = "../models/2020_2023/catboost_metadata.json"

In [3]:
print("Loading combined panel...")
df = pd.read_csv(DATA_PATH, low_memory=False)

df["date"] = pd.to_datetime(df["date"])
df["grid_id"] = df["grid_id"].astype("category")

target = "fire_today"

Loading combined panel...


In [4]:
df.head()

,date,grid_id,fire_today,lon,lat,w_lat,w_lon,T2M,PS,WS10M,...,pct_Pasture/Hay,pct_Cultivated Crops,pct_Woody Wetlands,pct_Emergent Herbaceous Wetlands,T2M_3d_mean,RH2M_3d_min,WS10M_3d_max,PRECTOT_7d_sum,fires_last_7d,fires_last_14d
0,2020-06-01,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.43,100.06,5.77,...,0.0,0.0,0.0,0.0,12.430000,90.21,5.77,0.08,NaN,NaN
1,2020-06-02,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.96,100.23,6.59,...,0.0,0.0,0.0,0.0,12.695000,89.99,6.59,0.08,0.0,0.0
2,2020-06-03,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.96,100.23,7.16,...,0.0,0.0,0.0,0.0,12.783333,89.99,7.16,0.08,0.0,0.0
3,2020-06-04,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.57,99.92,8.24,...,0.0,0.0,0.0,0.0,12.830000,89.65,8.24,0.08,0.0,0.0
4,2020-06-05,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.04,99.44,5.62,...,0.0,0.0,0.0,0.0,12.523333,89.65,8.24,0.18,0.0,0.0


In [5]:
print("Building BallTree for spatial neighbors...")

grid_meta = df[["grid_id", "lat", "lon"]].drop_duplicates().reset_index(drop=True)
coords_rad = np.radians(grid_meta[["lat", "lon"]].values)

tree = BallTree(coords_rad, metric="haversine")

Building BallTree for spatial neighbors...


In [6]:
print("Computing 8-nearest neighbors...")

K = 8 + 1   # +1 to include itself
dist_nn, idx_nn = tree.query(coords_rad, k=K)

# Map: grid_id → list of neighbor grid_ids
neighbor_map = {}
for i, g in enumerate(grid_meta["grid_id"]):
    neighbor_ids = grid_meta.iloc[idx_nn[i]]["grid_id"].tolist()
    neighbor_ids = [n for n in neighbor_ids if n != g]  # remove itself
    neighbor_map[g] = neighbor_ids


Computing 8-nearest neighbors...


In [7]:
print("Computing neighbor fire lag features...")

def add_neighbor_lag(df, lag_col):
    out_col = "neighbor_" + lag_col
    df[out_col] = 0

    for date, group in df.groupby("date"):
        # fast lookup table: grid_id -> lag value
        lag_map = dict(zip(group["grid_id"], group[lag_col]))

        values = []
        for gid in group["grid_id"]:
            neigh = neighbor_map[gid]
            total = sum(lag_map.get(n, 0) for n in neigh)
            values.append(total)

        df.loc[group.index, out_col] = values


lag_features = ["fires_last_7d", "fires_last_14d"]
for lf in lag_features:
    add_neighbor_lag(df, lf)

Computing neighbor fire lag features...


C:\Users\nilav\AppData\Local\Temp\ipykernel_23820\1424764654.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, na

In [8]:
print("Computing radius-based smoothing features (10 km)...")

radius_km = 10
radius_rad = radius_km / 6371.0  # Earth radius

# Query radius for all grid points
radius_neighbors = tree.query_radius(coords_rad, r=radius_rad)

# Smoothing function
def add_radius_mean(df, col, new_col):
    df[new_col] = 0
    for date, group in df.groupby("date"):
        # Build lookup for fast access
        value_map = dict(zip(group["grid_id"], group[col]))

        results = []
        for i, gid in enumerate(group["grid_id"]):
            neigh_idx = radius_neighbors[grid_meta.index[grid_meta["grid_id"] == gid][0]]
            neigh_ids = grid_meta.iloc[neigh_idx]["grid_id"]

            vals = [value_map.get(n, np.nan) for n in neigh_ids]
            results.append(np.nanmean(vals))

        df.loc[group.index, new_col] = results


radius_cols = {
    "T2M": "T2M_spatial_mean_10km",
    "RH2M": "RH2M_spatial_mean_10km",
    "WS10M": "WS10M_spatial_mean_10km",
    "ndvi": "ndvi_spatial_mean_10km"
}

# Add VPD proxy smoothing too
df["vpd_proxy"] = (1 - df["RH2M"] / 100) * df["T2M"]
radius_cols["vpd_proxy"] = "vpd_spatial_mean_10km"

for base, newname in radius_cols.items():
    add_radius_mean(df, base, newname)

Computing radius-based smoothing features (10 km)...


C:\Users\nilav\AppData\Local\Temp\ipykernel_23820\3447050861.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[12.527777777777779, 12.474, 12.430000000000001, 12.43, 12.43, 12.43, 12.430000000000001, 12.43, 12.625555555555556, 12.527777777777779, 12.47, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.43, 12.782, 12.693999999999999, 12.565384615384618, 12.430000000000003, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000003, 12.430000000000001, 12.430000000000001, 12.462, 12.87, 12.825999999999999, 12.76, 12.59923076923077, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000001, 12.430000000000003, 12.456666666666669, 12.526, 12.607777777777777, 12.678888888888

In [9]:
print("Engineering nonlinear features...")

df["hot_windy"] = df["T2M"] * df["WS10M"]
df["dry_windy"] = (100 - df["RH2M"]) * df["WS10M"]
df["fuel_moisture_proxy"] = df["ndvi"] * df["RH2M"]

optional_lags = ["fires_last_1d", "fires_last_3d"]
available_lags = [c for c in optional_lags if c in df.columns]

# Auto-detect NLCD columns
nlcd_cols = [c for c in df.columns if c.startswith("pct_")]

Engineering nonlinear features...


In [12]:
feature_cols = [
    # Weather
    "T2M", "RH2M", "WS10M", "PS", "PRECTOTCORR",
    "T2M_3d_mean", "RH2M_3d_min", "WS10M_3d_max", "PRECTOT_7d_sum",

    # Vegetation
    "ndvi", "ndvi_7d_mean",

    # Fires
    "fires_last_7d", "fires_last_14d"
] + available_lags + nlcd_cols + [

    # Nonlinear
    "vpd_proxy", "hot_windy", "dry_windy", "fuel_moisture_proxy",

    # Neighbor fire features
    "neighbor_fires_last_7d",
    "neighbor_fires_last_14d",

    # Radius-smoothed features
    "T2M_spatial_mean_10km",
    "RH2M_spatial_mean_10km",
    "WS10M_spatial_mean_10km",
    "ndvi_spatial_mean_10km",
    "vpd_spatial_mean_10km"
]

print(f"Using {len(feature_cols)} total features.")

Using 40 total features.


In [13]:
train = df[df["date"] < "2023-01-01"]
val   = df[df["date"] >= "2023-01-01"]

X_train = train[feature_cols]
y_train = train[target]

X_val = val[feature_cols]
y_val = val[target]

In [14]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / max(pos, 1)

print(f"Using class_weight = {scale:.2f}")


Using class_weight = 410.37


In [16]:
model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    learning_rate=0.05,
    depth=10,
    iterations=1500,
    l2_leaf_reg=3,
    random_state=42,
    thread_count=-1,
    class_weights=[1, scale],
    verbose=200,
    task_type='GPU',
    devices='0'
)

print("Training CatBoost model (with spatial features)...")
model.fit(X_train, y_train, eval_set=(X_val, y_val))

Training CatBoost model (with spatial features)...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8479774	best: 0.8479774 (0)	total: 62.8ms	remaining: 1m 34s
200:	test: 0.8454813	best: 0.8541666 (60)	total: 9.16s	remaining: 59.2s
400:	test: 0.8334516	best: 0.8541666 (60)	total: 20.1s	remaining: 55.2s
600:	test: 0.8274184	best: 0.8541666 (60)	total: 32.7s	remaining: 48.9s
800:	test: 0.8203975	best: 0.8541666 (60)	total: 44.1s	remaining: 38.5s
1000:	test: 0.8139128	best: 0.8541666 (60)	total: 56.4s	remaining: 28.1s
1200:	test: 0.8084840	best: 0.8541666 (60)	total: 1m 7s	remaining: 16.9s
1400:	test: 0.8038519	best: 0.8541666 (60)	total: 1m 19s	remaining: 5.63s
1499:	test: 0.8017414	best: 0.8541666 (60)	total: 1m 25s	remaining: 0us
bestTest = 0.8541666269
bestIteration = 60
Shrink model to first 61 iterations.


In [17]:
y_val_proba = model.predict_proba(X_val)[:, 1]

auc_roc = roc_auc_score(y_val, y_val_proba)
auc_pr = average_precision_score(y_val, y_val_proba)

print(f"\nROC-AUC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")


ROC-AUC: 0.8542
AUC-PR:  0.0797


In [18]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

best_idx = np.argmax(f1_scores)
best_threshold = float(thresholds[best_idx])

print(f"\nOptimal threshold (max F1): {best_threshold:.4f}")

y_pred_best = (y_val_proba >= best_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(y_val, y_pred_best, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_best))


Optimal threshold (max F1): 0.9582

Classification Report:
              precision    recall  f1-score   support

           0      0.998     0.995     0.996   2055406
           1      0.138     0.246     0.177      6516

    accuracy                          0.993   2061922
   macro avg      0.568     0.621     0.587   2061922
weighted avg      0.995     0.993     0.994   2061922

Confusion Matrix:
[[2045371   10035]
 [   4912    1604]]
